In [ ]:
%cd ../../

In [ ]:
import polars as pl
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm

# Load meal names

In [ ]:
dim_meals = pl.read_parquet("data/processed/dim_meals.parquet")
dim_meals.head()

# Encode

In [ ]:
names = (
    dim_meals
    .select(
        pl.col('id').alias('meal_id'),
        pl.col('names').alias('name')
    )
    .explode('name')
)
names.head()

In [ ]:
device = "mps"
tqdm.pandas()


EMBEDDING_MODEL_NAME = "jinaai/jina-embeddings-v3"
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
model     = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME, trust_remote_code=True, from_tf=False, use_flash_attn=False).to(device)

In [ ]:
outputs = model.encode(names['name'].to_list(), task="text-matching")

In [ ]:
names = (
    names
    .with_columns(
        pl.Series(outputs).alias('embedding')
    )
    .with_row_index('id')
)
names.head()

# Save

In [ ]:
path = "data/processed/dim_meal_names_embd.parquet"
names.write_parquet(path)